In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, count, when, isnan, countDistinct
from pyspark.sql import SparkSession

jar_paths = [
    "/home/jovyan/work/jars/delta-spark_2.12-3.1.0.jar",
    "/home/jovyan/work/jars/delta-storage-3.1.0.jar",
    "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar"
]
jars_string = ",".join(jar_paths)

spark = SparkSession.builder \
    .appName("LakehouseSetup_Offline") \
    .config("spark.jars", jars_string) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark Version: {spark.version}")
print("Connected successfully in Offline Mode! Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully in Offline Mode! Ready to build the Lakehouse.


Schema Enforcement

In [2]:
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException

table_path = "s3a://olist-data/silver/schema_demo_real_data"

print("==================================================")
print("STAGE 0: PREPARING REAL DATA FROM BRONZE")
print("==================================================")
# We load the actual bronze data, but limit it to 10 rows just for this fast demo
df_real_bronze = spark.read.format("delta").load("s3a://olist-data/bronze/olist_customers").limit(10)

print("==================================================")
print("STAGE 1: CREATING THE BASELINE SILVER TABLE")
print("==================================================")
# We establish our baseline contract using 4 core columns from the real data
df_baseline = df_real_bronze.select("customer_id", "customer_unique_id", "customer_city", "customer_state")

# We use overwriteSchema ONCE to establish the brand new table
df_baseline.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path)
print("Baseline table established with columns:", df_baseline.columns)


print("\n==================================================")
print("STAGE 2: UPSTREAM ADDS A COLUMN (mergeSchema)")
print("==================================================")
# The upstream team updates their CSV export to include the zip code
df_evolved = df_real_bronze.select(
    "customer_id", "customer_unique_id", "customer_city", "customer_state", "customer_zip_code_prefix"
)

print("Attempting to write real data with an EXTRA column: 'customer_zip_code_prefix'...")

try:
    # We use mergeSchema to tell Delta Lake to accept the new column
    df_evolved.write.format("delta").mode("append").option("mergeSchema", "true").save(table_path)
    print("SUCCESS: Table evolved safely. New column added.")
    
    # Show the first 2 rows to prove the column was appended
    spark.read.format("delta").load(table_path).show(2)
except AnalysisException as e:
    print(f"FAIL: {e}")


print("\n==================================================")
print("STAGE 3: UPSTREAM DELETES A COLUMN (Strict Enforcement)")
print("==================================================")
# The upstream team has a bug and their new CSV is missing 'customer_city'
df_missing_column = df_real_bronze.select(
    "customer_id", "customer_unique_id", "customer_state", "customer_zip_code_prefix"
)

print("Attempting to write real data MISSING a column: 'customer_city'...")

try:
    # We attempt a standard write (No mergeSchema, No overwriteSchema)
    # This activates Delta Lake's native Schema Enforcement
    df_missing_column.write.format("delta").mode("append").save(table_path)
    print("SUCCESS: Data written.")
except AnalysisException as e:
    print("PIPELINE CRASHED (INTENTIONAL)")
    print("Delta Lake caught the missing column and rejected the write!")
    print(f"Error Log: {str(e).split(';')[0]}") 
    
print("\n==================================================")
print("CONCLUSION: The pipeline successfully stopped bad data from entering the Lakehouse.")

STAGE 0: PREPARING REAL DATA FROM BRONZE
STAGE 1: CREATING THE BASELINE SILVER TABLE
Baseline table established with columns: ['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']

STAGE 2: UPSTREAM ADDS A COLUMN (mergeSchema)
Attempting to write real data with an EXTRA column: 'customer_zip_code_prefix'...
SUCCESS: Table evolved safely. New column added.
+--------------------+--------------------+-------------+--------------+------------------------+
|         customer_id|  customer_unique_id|customer_city|customer_state|customer_zip_code_prefix|
+--------------------+--------------------+-------------+--------------+------------------------+
|50edf1bb01f965f14...|58b5f3850e802b91a...|   santanesia|            RJ|                   27195|
|91489087a2f86dd34...|7b1af6c1d00a207d7...|       osasco|            SP|                    6112|
+--------------------+--------------------+-------------+--------------+------------------------+
only showing top 2 rows


STAGE 3: 

Time travel

In [3]:
df_history = spark.sql("DESCRIBE HISTORY delta.`s3a://olist-data/demo/olist_schema_test`")
df_history.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# 2. Time Travel to Past Version
print("\nTIME TRAVEL: Querying Version 0 (Original schema)")
df_past = spark.read.format("delta").option("versionAsOf", 0).load("s3a://olist-data/demo/olist_schema_test")
df_past.show()

# View Current Version
print("\nCURRENT STATE: Querying Version 1 (After Schema Evolution)")
df_present = spark.read.format("delta").option("versionAsOf", 1).load("s3a://olist-data/demo/olist_schema_test")
df_present.show()

# 3. Perform Rollback
print("\nROLLBACK: Restoring data to Version 0")
df_past.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("s3a://olist-data/demo/olist_schema_test")

print("Rollback successful. A new version is automatically appended to the history.")
spark.read.format("delta").load("s3a://olist-data/demo/olist_schema_test").show()

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|15     |2026-05-07 08:51:38    |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|14     |2026-05-07 08:51:35    |WRITE    |{mode -> Append, partitionBy -> []}   |
|13     |2026-05-07 08:51:34    |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|12     |2026-05-07 08:46:58    |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|11     |2026-05-07 08:39:52    |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|10     |2026-05-07 08:39:48    |WRITE    |{mode -> Append, partitionBy -> []}   |
|9      |2026-05-07 08:39:47    |WRITE    |{mode -> Append, partitionBy -> []}   |
|8      |2026-05-07 08:39:46    |WRITE    |{mode -> Overwrite, partitionBy -> []}|
|7      |2026-05-07 08:38:52.001|WRITE    |{mode -> Append, partitionBy -> []}   |
|6  